# Full-scale GPU experiment (shared CPU/GPU pipeline)

This notebook runs the same data preparation, feature definitions, target mapping, model configuration, and evaluation code as the 50K CPU sample. The only intended differences are data scale and requested training device.

**Important:** this notebook does not install RAPIDS/cuDF. The historical CSV-ingestion benchmark is separate; avoiding an unnecessary cuDF install prevents the dependency conflict seen in earlier Colab sessions.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

REPO_URL = 'https://github.com/xinghanchen-xc/Student-Wellbeing-Risk-Analytics.git'
REPO_DIR = Path('/content/Student-Wellbeing-Risk-Analytics')
if not (REPO_DIR / 'src' / '01_clean_data.py').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', Path.cwd())


In [ ]:
# Install repository dependencies. This intentionally does not install cudf/CuPy.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


## 1. Point to the full local dataset

The full file is not stored in GitHub. This default path follows the earlier Colab workflow; edit it if your 179 MB Parquet file is in a different Drive location. The input must contain the original source fields and `risk_level`. Existing engineered features and predictions are discarded and recomputed by the shared cleaner.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FULL_DATA_PATH = Path('/content/drive/MyDrive/Colab Notebooks/processed_features_1M.parquet')
if not FULL_DATA_PATH.exists():
    raise FileNotFoundError(f'Update FULL_DATA_PATH to the location of your full source file: {FULL_DATA_PATH}')
print('Input file:', FULL_DATA_PATH)
print('Size (MB):', round(FULL_DATA_PATH.stat().st_size / (1024 * 1024), 2))


## 2. Rebuild the full dataset using canonical features

The cleaner accepts Parquet, CSV, CSV.GZ, or ZIP. It retains only source input fields, the source target, and optional descriptive outcome columns. It does not pass `dropout_risk`, `burnout_score`, `mental_health_index`, old target encodings, or previous prediction fields into the model.


In [ ]:
PROCESSED_DIR = Path('outputs/full_1m/processed')
subprocess.run([
    sys.executable, 'src/01_clean_data.py',
    '--input', str(FULL_DATA_PATH),
    '--out-dir', str(PROCESSED_DIR),
    '--dashboard-sample-size', '100000',
], check=True)
CLEANED_PATH = PROCESSED_DIR / 'student_burnout_cleaned.csv.gz'
assert CLEANED_PATH.exists(), CLEANED_PATH
print('Prepared data:', CLEANED_PATH)


## 3. Train and evaluate XGBoost on GPU

The script uses the same stratified 80/20 split, fixed random seed, explicit feature allowlist, balanced sample weights, and model parameters as the 50K CPU run. If CUDA training fails, the script stops instead of silently retrying on CPU. After training, inference is explicitly run on CPU because the sklearn preprocessing stage produces NumPy arrays on CPU; the report records both devices.


In [ ]:
RUN_DIR = Path('outputs/full_1m/run')
subprocess.run([
    sys.executable, 'src/02_train_model_generate_results.py',
    '--cleaned', str(CLEANED_PATH),
    '--out-dir', str(RUN_DIR),
    '--device', 'cuda',
    '--test-size', '0.2',
    '--dashboard-sample-size', '100000',
    '--shap-sample-size', '300',
], check=True)


In [ ]:
metrics_path = RUN_DIR / 'reports' / 'model_metrics.json'
with metrics_path.open(encoding='utf-8') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))


## Interpretation notes

- Quote only the metrics generated by this run; do not reuse historical 88% / 0.87 values unless this leakage-safe pipeline actually reproduces them.
- Review per-class precision/recall and the confusion matrix, especially for the rare High-risk class.
- The Kaggle dataset is described by its publisher as synthetic. These results do not establish real-world or clinical validity.
- The 50K sample and full-scale run share feature code and model logic, but their metric values are not directly comparable as a pure CPU-versus-GPU performance test because the data sizes differ.
